In [1]:
import os
import re
from typing import TypedDict

from dotenv import load_dotenv
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_groq import ChatGroq
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import END, START, StateGraph
from langgraph.types import Command, interrupt
from pydantic import BaseModel

load_dotenv()

groq_api_key = os.getenv('GROQ_API_KEY')

C:\Users\HP\AppData\Local\Temp\ipykernel_22784\3544584422.py:6: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


In [2]:
# -----------------------------
# Data + Index
# -----------------------------
docs = (
    PyPDFLoader("../data/documents/book1.pdf").load()
    + PyPDFLoader("../data/documents/book2.pdf").load()
    + PyPDFLoader("../data/documents/book3.pdf").load()
)

fontTools is required to fully parse the encoding of a CFF Type1 font in font dictionary {'/Subtype': '/Type1', '/FontDescriptor': IndirectObject(25, 0, 2658708342832), '/LastChar': 1, '/Widths': [833], '/BaseFont': '/NJBOIP+MathematicalPi-Three', '/FirstChar': 1, '/Encoding': IndirectObject(26, 0, 2658708342832), '/Type': '/Font'}, but is not installed. Consider installing fontTools if you encounter encoding problems.
fontTools is required to fully parse the encoding of a CFF Type1 font in font dictionary {'/Subtype': '/Type1', '/FontDescriptor': IndirectObject(743, 0, 2658708342832), '/LastChar': 2, '/Widths': [778, 778], '/BaseFont': '/NJDFHF+MSAM10', '/FirstChar': 1, '/Encoding': IndirectObject(744, 0, 2658708342832), '/Type': '/Font'}, but is not installed. Consider installing fontTools if you encounter encoding problems.
fontTools is required to fully parse the encoding of a CFF Type1 font in font dictionary {'/Subtype': '/Type1', '/FontDescriptor': IndirectObject(743, 0, 2658708

In [3]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=900, chunk_overlap=150)
chunks = text_splitter.split_documents(docs)
for d in chunks:
    d.page_content = d.page_content.encode("utf-8", "ignore").decode("utf-8", "ignore")

In [4]:
embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vector_store = FAISS.from_documents(chunks, embedding_model)
retriever = vector_store.as_retriever(search_type="similarity", search_kwargs={"k": 4})

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [5]:
llm = ChatGroq(model="openai/gpt-oss-120b", groq_api_key=groq_api_key, temperature=0)

In [6]:
UPPER_TH = 0.7
LOWER_TH = 0.3

In [7]:
# -----------------------------
# State
# -----------------------------
class State(TypedDict):
    question: str

    docs: list[Document]
    good_docs: list[Document]

    verdict: str
    reason: str

    strips: list[str]
    kept_strips: list[str]
    refined_context: str

    web_query: str
    
    web_docs: list[Document]
    
    human_approval: str

    answer: str
    

In [8]:
# -----------------------------
# Retrieve
# -----------------------------
def retrieve_node(state: State) -> State:
    q = state["question"]
    return {"docs": retriever.invoke(q)}

In [9]:
# -----------------------------
# Score-based doc evaluator
# -----------------------------
class DocEvalScore(BaseModel):
    score: float
    reason: str


doc_eval_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            (
                "You are a strict retrieval evaluator for RAG.\n"
                "You will be given ONE retrieved chunk and a question.\n"
                "Return a relevance score in [0.0, 1.0].\n"
                "- 1.0: chunk alone is sufficient to answer fully/mostly\n"
                "- 0.0: chunk is irrelevant\n"
                "Be conservative with high scores.\n"
                "Also return a short reason.\n"
                "Output JSON only."
            ),
        ),
        ("human", "Question: {question}\n\nChunk:\n{chunk}"),
    ]
)


doc_eval_llm = llm.with_structured_output(DocEvalScore, method= "json_schema")
doc_eval_chain = doc_eval_prompt | doc_eval_llm


def eval_each_doc_node(state: State) -> State:
    q = state["question"]
    scores: list[float] = []
    good: list[Document] = []

    for d in state["docs"]:
        out = doc_eval_chain.invoke({"question": q, "chunk": d.page_content})
        scores.append(out.score)

        # Keep any doc above LOWER_TH as "weakly relevant"
        if out.score > LOWER_TH:
            good.append(d)

    # CORRECT: at least one doc > UPPER_TH
    if any(s > UPPER_TH for s in scores):
        return {
            "good_docs": good,
            "verdict": "CORRECT",
            "reason": f"At least one retrieved chunk scored > {UPPER_TH}.",
        }

    # INCORRECT: all docs < LOWER_TH
    if len(scores) > 0 and all(s < LOWER_TH for s in scores):
        return {
            "good_docs": [],
            "verdict": "INCORRECT",
            "reason": f"All retrieved chunks scored < {LOWER_TH}.",
        }

    # AMBIGUOUS: otherwise
    return {
        "good_docs": good,
        "verdict": "AMBIGUOUS",
        "reason": f"No chunk scored > {UPPER_TH}, but not all were < {LOWER_TH}.",
    }

In [10]:
# -----------------------------
# Sentence-level DECOMPOSER
# -----------------------------
def decompose_to_sentences(text: str) -> list[str]:
    text = re.sub(r"\s+", " ", text).strip()
    sentences = re.split(r"(?<=[.!?])\s+", text)
    return [s.strip() for s in sentences if len(s.strip()) > 20]


# -----------------------------
# FILTER (LLM judge)
# -----------------------------
class KeepOrDrop(BaseModel):
    keep: bool


filter_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            (
                "You are a strict relevance filter.\n"
                "Return keep=true only if the sentence directly helps answer the question.\n"
                "Use ONLY the sentence. Output JSON only."
            ),
        ),
        ("human", "Question: {question}\n\nSentence:\n{sentence}"),
    ]
)

filter_llm = llm.with_structured_output(KeepOrDrop, method='json_schema')
filter_chain = filter_prompt | llm.with_structured_output(KeepOrDrop)


# -----------------------------
# Knowledge refinement
# (CORRECT => internal only)
# (INCORRECT => web only)
# (AMBIGUOUS => internal + web)
# -----------------------------
def refine(state: State) -> State:
    q = state["question"]

    if state.get("verdict") == "CORRECT":
        docs_to_use = state["good_docs"]
    elif state.get("verdict") == "INCORRECT":
        docs_to_use = state["web_docs"]
    else:  # AMBIGUOUS
        docs_to_use = state["good_docs"] + state["web_docs"]

    context = "\n\n".join(d.page_content for d in docs_to_use).strip()

    strips = decompose_to_sentences(context)

    kept: list[str] = []
    for s in strips:
        if filter_chain.invoke({"question": q, "sentence": s}).keep:
            kept.append(s)

    refined_context = "\n".join(kept).strip()

    return {
        "strips": strips,
        "kept_strips": kept,
        "refined_context": refined_context,
    }

In [11]:
# -----------------------------
# Query rewrite for web search
# -----------------------------
class WebQuery(BaseModel):
    query: str


rewrite_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            (
                "Rewrite the user question into a web search query composed of keywords.\n"
                "Rules:\n"
                "- Keep it short (6–14 words).\n"
                "- If the question implies recency (e.g., recent/latest/last week/last month), "
                "add a constraint like (last 30 days).\n"
                "- Do NOT answer the question.\n"
                "- Return JSON with a single key: query"
            ),
        ),
        ("human", "Question: {question}"),
    ]
)

rewrite_llm = llm.with_structured_output(WebQuery, method='json_schema')

rewrite_chain = rewrite_prompt | rewrite_llm


def rewrite_query_node(state: State) -> State:
    out = rewrite_chain.invoke({"question": state["question"]})
    return {"web_query": out.query}


# -----------------------------
# Web search node: uses web_query
# -----------------------------
tavily = TavilySearchResults(max_results=5)


def web_search_node(state: State) -> State:
    q = state.get("web_query") or state["question"]
    results = tavily.invoke({"query": q})

    web_docs: list[Document] = []
    for r in results or []:
        title = r.get("title", "")
        url = r.get("url", "")
        content = r.get("content", "") or r.get("snippet", "")
        text = f"TITLE: {title}\nURL: {url}\nCONTENT:\n{content}"
        web_docs.append(Document(page_content=text, metadata={"url": url, "title": title}))

    return {"web_docs": web_docs}

C:\Users\HP\AppData\Local\Temp\ipykernel_22784\4052662389.py:39: LangChainDeprecationWarning: The class `TavilySearchResults` was deprecated in LangChain 0.3.25 and will be removed in 1.0. An updated version of the class exists in the `langchain-tavily package and should be used instead. To use it run `pip install -U `langchain-tavily` and import as `from `langchain_tavily import TavilySearch``.
  tavily = TavilySearchResults(max_results=5)


In [12]:
# -----------------------------
# Generate
# -----------------------------
answer_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            (
                "You are a helpful ML tutor. Answer ONLY using the provided context.\n"
                "If the context is empty or insufficient, say: 'I don't know.'"
            ),
        ),
        ("human", "Question: {question}\n\nContext:\n{context}"),
    ]
)


def generate(state: State) -> State:
    out = (answer_prompt | llm).invoke({"question": state["question"], "context": state["refined_context"]})
    return {"answer": out.content}

In [13]:
# -----------------------------
# Routing
# CORRECT => refine
# INCORRECT / AMBIGUOUS => rewrite -> web_search -> refine -> generate
# -----------------------------
def route_after_eval(state: State) -> str:
    if state["verdict"] == "CORRECT":
        return "refine"
    else:
        return "human_approval"

In [14]:
def human_approval_node(state: State) -> State:

    decision = interrupt({
        "message": "The retrieved documents are insufficient or ambiguous.",
        "question": state["question"],
        "reason": state["reason"]
    })

    return {
        "human_approval": decision
    }

In [15]:
def route_after_human(state: State) -> str:

    if str(state["human_approval"]).lower() in ["yes", "y", "true", "approve", "approved"]:
        
        return "rewrite_query"

    return "reject"

In [16]:
def human_reject_node(state: State) -> State:
    return {
        "answer": (
            "The retrieved documents were insufficient to answer the question, "
            "and web search was not approved."
        )
    }

In [17]:
# -----------------------------
# Build graph
# -----------------------------
g = StateGraph(State)

# Nodes
g.add_node("retrieve", retrieve_node)
g.add_node("eval_each_doc", eval_each_doc_node)
g.add_node("human_approval", human_approval_node)
g.add_node("human_reject", human_reject_node)
g.add_node("rewrite_query", rewrite_query_node)
g.add_node("web_search", web_search_node)
g.add_node("refine", refine)
g.add_node("generate", generate)

# Initial flow
g.add_edge(START, "retrieve")
g.add_edge("retrieve", "eval_each_doc")

# Route based on document evaluation
g.add_conditional_edges("eval_each_doc",route_after_eval, {"refine": "refine", "human_approval": "human_approval"})

# Human approval routing
g.add_conditional_edges("human_approval",route_after_human, {"rewrite_query": "rewrite_query", "reject": "human_reject"})

g.add_edge("human_reject", END)

# Web-search path
g.add_edge("rewrite_query", "web_search")
g.add_edge("web_search", "refine")

# Final answer
g.add_edge("refine", "generate")
g.add_edge("generate", END)

# Checkpointer
memory = MemorySaver()

app = g.compile(checkpointer=memory)
print(app)

In [18]:
# -----------------------------
# Run example
# -----------------------------
config = {"configurable": {"thread_id": "crag-demo-1"}}

res = app.invoke({
    "question": "Batch normalization vs layer normalization",
    "docs": [],
    "good_docs": [],
    "verdict": "",
    "reason": "",
    "strips": [],
    "kept_strips": [],
    "refined_context": "",
    "web_query": "",
    "web_docs": [],
    "answer": "",
    "human_approval": "",
}, config=config)

print("VERDICT:", res["verdict"])
print("REASON:", res["reason"])

print("WEB_QUERY:", res["web_query"])
print("\nOUTPUT:\n", res["answer"])

VERDICT: AMBIGUOUS
REASON: No chunk scored > 0.7, but not all were < 0.3.
WEB_QUERY: 

OUTPUT:
 


In [19]:
config = {"configurable": {"thread_id": "crag-demo-1"}}

res = app.invoke({
    "question": "What is the current stock price of NVIDIA?",
    "docs": [],
    "good_docs": [],
    "verdict": "",
    "reason": "",
    "strips": [],
    "kept_strips": [],
    "refined_context": "",
    "web_query": "",
    "web_docs": [],
    "answer": "",
    "human_approval": "",
}, config=config)

print(res)

{'question': 'What is the current stock price of NVIDIA?', 'docs': [Document(id='83474336-1127-4e03-945d-090ed7b49d23', metadata={'producer': 'Antenna House PDF Output Library 6.2.609 (Linux64)', 'creator': 'AH CSS Formatter V6.2 MR4 for Linux64 : 6.2.6.18551 (2014/09/24 15:00JST)', 'creationdate': '2017-03-10T21:55:34+00:00', 'author': 'Aurélien Géron', 'moddate': '2017-07-29T21:43:02+08:00', 'title': 'Hands-On Machine Learning with Scikit-Learn and TensorFlow', 'trapped': '/False', 'source': '../data/documents/book3.pdf', 'total_pages': 564, 'page': 337, 'page_label': '316'}, page_content='$ nvidia-smi\nWed Sep 16 09:50:03 2016\n+------------------------------------------------------+\n| NVIDIA-SMI 352.63     Driver Version: 352.63         |\n|-------------------------------+----------------------+----------------------+\n| GPU  Name        Persistence-M| Bus-Id        Disp.A | Volatile Uncorr. ECC |\n| Fan  Temp  Perf  Pwr:Usage/Cap|         Memory-Usage | GPU-Util  Compute M. |\n|=

In [20]:
res = app.invoke(
    Command(resume="yes"),
    config={"configurable": {"thread_id": "crag-demo-1"}}
)

print(res)

{'question': 'What is the current stock price of NVIDIA?', 'docs': [Document(id='83474336-1127-4e03-945d-090ed7b49d23', metadata={'producer': 'Antenna House PDF Output Library 6.2.609 (Linux64)', 'creator': 'AH CSS Formatter V6.2 MR4 for Linux64 : 6.2.6.18551 (2014/09/24 15:00JST)', 'creationdate': '2017-03-10T21:55:34+00:00', 'author': 'Aurélien Géron', 'moddate': '2017-07-29T21:43:02+08:00', 'title': 'Hands-On Machine Learning with Scikit-Learn and TensorFlow', 'trapped': '/False', 'source': '../data/documents/book3.pdf', 'total_pages': 564, 'page': 337, 'page_label': '316'}, page_content='$ nvidia-smi\nWed Sep 16 09:50:03 2016\n+------------------------------------------------------+\n| NVIDIA-SMI 352.63     Driver Version: 352.63         |\n|-------------------------------+----------------------+----------------------+\n| GPU  Name        Persistence-M| Bus-Id        Disp.A | Volatile Uncorr. ECC |\n| Fan  Temp  Perf  Pwr:Usage/Cap|         Memory-Usage | GPU-Util  Compute M. |\n|=

In [21]:
config_no = {
    "configurable": {
        "thread_id": "crag-demo-no"
    }
}

res = app.invoke({
    "question": "What is the current stock price of NVIDIA?",
    "docs": [],
    "good_docs": [],
    "verdict": "",
    "reason": "",
    "strips": [],
    "kept_strips": [],
    "refined_context": "",
    "web_query": "",
    "web_docs": [],
    "answer": "",
    "human_approval": "",
}, config=config_no)

print(res)

{'question': 'What is the current stock price of NVIDIA?', 'docs': [Document(id='83474336-1127-4e03-945d-090ed7b49d23', metadata={'producer': 'Antenna House PDF Output Library 6.2.609 (Linux64)', 'creator': 'AH CSS Formatter V6.2 MR4 for Linux64 : 6.2.6.18551 (2014/09/24 15:00JST)', 'creationdate': '2017-03-10T21:55:34+00:00', 'author': 'Aurélien Géron', 'moddate': '2017-07-29T21:43:02+08:00', 'title': 'Hands-On Machine Learning with Scikit-Learn and TensorFlow', 'trapped': '/False', 'source': '../data/documents/book3.pdf', 'total_pages': 564, 'page': 337, 'page_label': '316'}, page_content='$ nvidia-smi\nWed Sep 16 09:50:03 2016\n+------------------------------------------------------+\n| NVIDIA-SMI 352.63     Driver Version: 352.63         |\n|-------------------------------+----------------------+----------------------+\n| GPU  Name        Persistence-M| Bus-Id        Disp.A | Volatile Uncorr. ECC |\n| Fan  Temp  Perf  Pwr:Usage/Cap|         Memory-Usage | GPU-Util  Compute M. |\n|=

In [22]:
res = app.invoke(
    Command(resume="no"),
    config=config_no
)

print(res["answer"])

The retrieved documents were insufficient to answer the question, and web search was not approved.


In [23]:
def run_crag(question: str, thread_id: str = "crag-interactive"):
    
    config = {"configurable": {"thread_id": thread_id}}

    # Initial state
    initial_state = {
        "question": question,
        "docs": [],
        "good_docs": [],
        "verdict": "",
        "reason": "",
        "strips": [],
        "kept_strips": [],
        "refined_context": "",
        "web_query": "",
        "web_docs": [],
        "answer": "",
        "human_approval": "",
    }

    # Start the graph
    result = app.invoke(initial_state,config=config)

    # Keep running until graph finishes
    while "__interrupt__" in result:

        interrupt_data = result["__interrupt__"][0].value

        print("\n" + "=" * 60)
        print("HUMAN APPROVAL REQUIRED")
        print("=" * 60)

        print("\nQuestion:")
        print(interrupt_data["question"])

        print("\nReason:")
        print(interrupt_data["reason"])

        print("\n" + "-" * 60)

        while True:
            user_input = input("Do you want to search the web? (yes/no): ").strip().lower()

            if user_input in ["yes", "y", "no", "n"]:
                break

            print("Please enter only yes or no.")

        # Resume graph with user's decision
        result = app.invoke(
            Command(resume=user_input),
            config=config
        )

    print("\n" + "=" * 60)
    print("FINAL ANSWER")
    print("=" * 60)

    print(result["answer"])

    return result

In [24]:
result = run_crag(
    "What is the current stock price of NVIDIA?"
)


HUMAN APPROVAL REQUIRED

Question:
What is the current stock price of NVIDIA?

Reason:
All retrieved chunks scored < 0.3.

------------------------------------------------------------

FINAL ANSWER
The current NVIDIA (NVDA) stock price is **$222.27** per share.


In [28]:
result = run_crag(
    "What is machine learning?"
)


FINAL ANSWER
Machine learning is the science (and art) of programming computers so they can improve at a task by learning from data rather than by being given explicit rules. In formal terms, a computer program learns from experience E with respect to some task T and performance measure P if its performance on T, as measured by P, improves with E. In practice, a machine‑learning system gathers a training set of data, feeds it to a learning algorithm, and adjusts model parameters so that it can make good predictions on both the training data and new, unseen cases.
